# Hoi! Challenge — T1 Articulation Estimation: HOI-DETR Batch Inference

**Goal of this notebook:** run [HOI-DETR](https://github.com/AhmadDarKhalil/HOI-DETR) over every frame of every
object sequence in the Hoi! dataset eval split, and save out per-frame detections
(hand box, 1st-object box = the moving door/drawer panel, 2nd-object box = the
static cabinet body, and the hand↔object interaction link).

**Why this matters for T1 (Articulation Estimation):**
- The **1st-object box** localizes the panel that actually moves (the door/drawer front).
  We'll later seed a point tracker inside this region to recover its 2D trajectory.
- The **2nd-object box** (the static cabinet body) gives us a *stationary reference region*.
  Since the camera is head-mounted and moving, we need to separate camera-induced
  motion from object-induced motion — tracking points on the static cabinet lets us
  estimate camera motion and subtract it out.
- This notebook only produces **detections** (bounding boxes per frame). Point tracking,
  line/arc fitting for the revolute-vs-prismatic classification (Level 1), and 3D axis
  estimation (Level 2) are separate downstream steps, not covered here.

**Environment:** this notebook is written for **Kaggle Notebooks** with a **GPU T4 x2**
or **P100** accelerator enabled (Settings → Accelerator, in the notebook sidebar).
HOI-DETR needs an old, pinned stack (Python-compatible torch 1.11 + mmcv-full 1.5.0
compiled for CUDA 11.3), so we install everything explicitly rather than relying on
Kaggle's default pre-installed versions.

**Before running:** upload your dataset as a private Kaggle Dataset (zip of your
`kitchen_9/`, `kitchen_16/` folders) and attach it to this notebook via
**Add Data** in the sidebar. It will then be mounted at `/kaggle/input/<your-dataset-name>/`.
Update `DATASET_ROOT` in the config cell below to match.


## 1. Environment setup

We pin exact versions to match what HOI-DETR/Co-DETR was built and tested against.
This step typically takes several minutes — mostly downloading the correct CUDA-matched
wheels for `torch` and `mmcv-full`.

> Note: if a version conflict error appears here, restart the notebook session
> (Kaggle: "Factory reset" from the kernel menu) and re-run from the top — this avoids
> partially-upgraded package states from Kaggle's preinstalled environment.


In [ ]:
# Confirm GPU is visible before spending time installing anything
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


In [ ]:
%%capture
# Pin torch to the CUDA 11.3 build HOI-DETR/Co-DETR was tested against
!pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0+cu113 \
    --extra-index-url https://download.pytorch.org/whl/cu113

# mmcv-full compiled against the same torch/cuda combo
!pip install mmcv-full==1.5.0 \
    -f https://download.openmmlab.com/mmcv/dist/cu113/torch1.11/index.html

# Remaining pinned dependencies from the HOI-DETR README
!pip install timm==0.6.13 fairscale==0.4.6 scipy==1.7.3 yapf==0.40.1 \
    opencv-python numpy==1.21.6 pycocotools


In [ ]:
# Sanity-check the installed versions before going further
import torch, mmcv
print("torch:", torch.__version__)
print("mmcv:", mmcv.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


## 2. Clone HOI-DETR and download the checkpoint

The repo is MIT-licensed and builds on Co-DETR / MMDetection. We install it in
editable mode (`pip install -e .`) so its custom modules are importable.


In [ ]:
%cd /kaggle/working
!git clone https://github.com/AhmadDarKhalil/HOI-DETR.git
%cd HOI-DETR
!pip install -e .


In [ ]:
# Download the released checkpoint (ViT-L/16, trained on Hands23)
# Check the repo README / releases page for the current checkpoint URL if this changes.
%cd /kaggle/working/HOI-DETR
!mkdir -p checkpoints
!wget -O checkpoints/epoch_5.pth <PASTE_CHECKPOINT_URL_FROM_REPO_README_OR_HF>


> **Action needed:** the HOI-DETR README links the checkpoint via a Hugging Face
> `wget` command — copy the exact URL from their README (it may change over time)
> and paste it into the cell above in place of `<PASTE_CHECKPOINT_URL_FROM_REPO_README_OR_HF>`.


## 3. Configuration

Point this at your uploaded Kaggle dataset. Update `DATASET_ROOT` to match the
exact path Kaggle mounts your dataset at (visible in the sidebar under "Input" once attached).


In [ ]:
import os

# --- EDIT THESE ---
DATASET_ROOT = "/kaggle/input/hoi-dataset-eval-split"   # <- change to your actual dataset slug
OUTPUT_ROOT = "/kaggle/working/detections"               # per-frame detection results go here
CONF_THRESHOLD = 0.3                                     # lower this if boxes are missing on your frames
# -------------------

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Discover object sequences: expects DATASET_ROOT/kitchen_9/object_*/rgb/*.jpg (adjust glob if your structure differs)
import glob
sequence_dirs = sorted(glob.glob(os.path.join(DATASET_ROOT, "kitchen_*", "object_*")))
print(f"Found {len(sequence_dirs)} object sequences")
print(sequence_dirs[:3])


## 4. Load the model

This loads the Co-DETR config + checkpoint via `mmdet`'s inference API.
Adjust the config path below to whichever config file in the repo corresponds
to the released checkpoint (check `configs/` in the cloned repo — the README
names the specific config used for `epoch_5.pth`).


In [ ]:
from mmdet.apis import init_detector, inference_detector

CONFIG_PATH = "/kaggle/working/HOI-DETR/configs/co_detr/<CONFIG_FILE_FROM_REPO>.py"  # <- fill in exact filename
CHECKPOINT_PATH = "/kaggle/working/HOI-DETR/checkpoints/epoch_5.pth"

model = init_detector(CONFIG_PATH, CHECKPOINT_PATH, device="cuda:0")
print("Model loaded.")


## 5. Batch inference over all frames

For each sequence, run detection on every frame and save out a JSON with:
- hand box(es)
- 1st-object box (the moving door/drawer panel) — used later to seed point tracking
- 2nd-object box (static cabinet body) — used later as the camera-motion reference
- the hand↔object interaction link/confidence

This can take a while depending on frame counts — the `tqdm` progress bar helps
you track it against your Kaggle GPU-hour quota (30 hrs/week, 12 hrs/session max).


In [ ]:
import json
import cv2
from tqdm import tqdm

def parse_detections(result, conf_thresh=CONF_THRESHOLD):
    """
    Convert raw mmdet inference_detector output into a simple dict.
    NOTE: adjust this parsing logic once you inspect the actual output format
    of this specific model — HOI-DETR's output head (hand / 1st obj / 2nd obj /
    interaction link) may not follow the standard single-class-list mmdet format,
    since it's a multi-role detector. Check the repo's demo script
    (referenced on their project page) for the exact expected parsing, and adapt
    this function to match before trusting the saved JSONs.
    """
    detections = {"hands": [], "first_objects": [], "second_objects": [], "links": []}
    # --- placeholder structure; replace with real parsing based on repo's demo.py ---
    return detections


for seq_dir in tqdm(sequence_dirs, desc="Sequences"):
    rgb_dir = os.path.join(seq_dir, "rgb")
    if not os.path.isdir(rgb_dir):
        continue

    frame_files = sorted(os.listdir(rgb_dir))
    seq_name = "_".join(seq_dir.rstrip("/").split("/")[-2:])  # e.g. kitchen_9_object_1
    out_path = os.path.join(OUTPUT_ROOT, f"{seq_name}.json")

    seq_results = {}
    for fname in frame_files:
        frame_path = os.path.join(rgb_dir, fname)
        img = cv2.imread(frame_path)
        result = inference_detector(model, img)
        seq_results[fname] = parse_detections(result)

    with open(out_path, "w") as f:
        json.dump(seq_results, f)

print("Done. Detections saved to", OUTPUT_ROOT)


> **Important:** the `parse_detections` function above is a placeholder.
> HOI-DETR's output isn't a standard single-list mmdet detector output — it jointly
> predicts hands, 1st/2nd objects, and their interaction links, so before trusting
> these results, inspect the raw structure of `inference_detector(model, img)` on
> one test image and adapt the parsing to correctly separate the three box types
> and the link confidences. The repo's own demo/inference script (used for their
> HF Space demo) is the best reference for the exact output format.


## 6. Quick visual sanity check

Before trusting the full batch run, visualize detections on a handful of frames —
same idea as the demo you tested on Hugging Face, but run locally here so you can
confirm the pinned-environment install reproduces the same quality of detections.


In [ ]:
import matplotlib.pyplot as plt
import random

sample_seq = random.choice(sequence_dirs)
rgb_dir = os.path.join(sample_seq, "rgb")
frames = sorted(os.listdir(rgb_dir))
sample_frame = frames[len(frames) // 2]  # a middle frame, likely mid-interaction

img = cv2.cvtColor(cv2.imread(os.path.join(rgb_dir, sample_frame)), cv2.COLOR_BGR2RGB)
result = inference_detector(model, cv2.imread(os.path.join(rgb_dir, sample_frame)))

plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.title(f"{sample_seq} — {sample_frame}")
plt.axis("off")
plt.show()

# TODO once parse_detections is filled in: overlay hand / 1st-object / 2nd-object
# boxes here with cv2.rectangle or matplotlib patches, using different colors per role.


## 7. Download results

Kaggle notebooks reset their working directory between sessions, so zip and
download (or save as a Kaggle Dataset output) before your session ends.


In [ ]:
!zip -r /kaggle/working/detections.zip /kaggle/working/detections
print("Zipped — download via the Kaggle 'Output' pane on the right sidebar.")


## Next steps (not covered in this notebook)

1. Fix `parse_detections` against the real HOI-DETR output format.
2. For each sequence, use the 1st-object box to seed a point tracker (e.g. KLT via
   OpenCV, or CoTracker) across frames — this gives the 2D trajectory of the moving panel.
3. Use points in the 2nd-object (cabinet) box as a static reference to estimate and
   subtract out camera-induced motion.
4. Classify revolute vs. prismatic (T1 Level 1) by fitting a line vs. an arc to the
   corrected trajectory and comparing residuals.
5. Estimate the 3D joint axis (T1 Level 2) using the known camera intrinsics to lift
   2D trajectories into 3D reasoning (pinhole ray equations, no distortion correction needed
   since `distortion: [0,0,0,0,0]` for this dataset).
6. Package predictions into `predictions.json` per the Codabench T1 submission format
   and zip with the file at the root level (not nested, not named `ground_truth.json`).
